In [1]:
import os

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
import pandas as pd


load_dotenv()  # Loads variables from .env into os.environ

True

In [5]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Equivalent explicit version (useful if you manage multiple keys):
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [6]:
response = client.chat.completions.create(
    model="llama3.2:latest", # Ensure this matches 'ollama list'
    messages=[
        {"role": "user", "content": "What is RAG (Retrieval-Augmented Generation)? Explain in 2-3 sentences."}
    ]
)

In [7]:
print(response.choices[0].message.content)

RAG (Retrieval-Augmented Generation) is a natural language processing technique that combines retrieval and generation tasks to improve the accuracy and efficiency of text generation models. This approach enables models to efficiently retrieve relevant information from a large dataset and then generate new text based on that retrieved knowledge, often leading to more coherent and informative results. By leveraging both retrieving and generating capabilities, RAG systems can learn to better understand context and adapt to different scenarios.


In [9]:
model_name = os.environ.get("OLLAMA_MODEL", "llama3.2")

response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a concise Python tutor."},
        {"role": "user", "content": "Explain list comprehensions with one example."}
    ],
    stream=True, 
)
print("AI Response: ", end="")
for chunk in response:
    # Standard OpenAI SDK uses choices[0].delta.content for streaming
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
print() # Final newline

AI Response: **List Comprehensions**

A list comprehension is a concise way to create lists in Python by performing an operation on each element of an existing iterable (such as a list, tuple, or set).

**Basic Syntax**
-----------------

Here's the basic syntax:
```python
[ expression for variable in iterable [if condition] ]
```
*   `expression` is the value you want to compute for each element.
*   `variable` is the temporary variable used to access elements of the iterable.
*   `iterable` is the list, tuple, or set you want to manipulate.
*   `[condition]` is an optional filter clause that applies only to selected elements.

**Example: Squaring Numbers**
-----------------------------

Suppose we have a list of numbers and we want to create a new list containing their squares. We can use a list comprehension:
```python
numbers = [1, 2, 3, 4, 5]
squares = [x ** 2 for x in numbers]

print(squares)  # Output: [1, 4, 9, 16, 25]
```
In this example, `x` is the temporary variable used to 

In [12]:

messages = [
    {"role": "system", "content": "You are a coding assistant. Be concise."},
    {"role": "user", "content": "What is a Python generator?"}
]
response_1 = client.chat.completions.create(
    model=model_name,
    messages=messages
)

# Store the AI's answer in our history
ai_answer_1 = response_1.choices[0].message.content
messages.append({"role": "assistant", "content": ai_answer_1})

print(f"Turn 1: {ai_answer_1[:200]}...\n")


# --- Turn 2: Follow-up (The model 'remembers' Turn 1) ---
messages.append({"role": "user", "content": "Show me a practical example of one."})

response_2 = client.chat.completions.create(
    model=model_name,
    messages=messages  # Sending the whole list provides the "Memory"
)

print(f"Turn 2: {response_2.choices[0].message.content}")

Turn 1: A Python generator is a special type of function that can be paused and resumed during its execution, allowing it to generate a series of values on-the-fly rather than computing them all at once.

Her...

Turn 2: **Reading and Processing Large Files using Generators**

Suppose we have a large text file containing comma-separated IDs and names, and we want to process each line without loading the entire file into memory.

```python
def read_large_file(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            # Split the line into ID and name
            id_name = line.strip().split(',')
            yield (int(id_name[0]), id_name[1])

# Usage example
file_path = 'large_data.txt'
for record in read_large_file(file_path):
    print(record)
```

In this example, `read_large_file` is a generator function that reads the file line by line, yields each tuple of ID and name, and returns it immediately without loading the entire file into memory. The callin

In [13]:
class CodeReview(BaseModel):
    """Schema for structured code review output."""
    language: str
    summary: str
    issues: list[str]
    severity: str  # e.g., "low", "medium", "high"

In [14]:
response = client.beta.chat.completions.parse(
    model=model_name,
    messages=[
        {"role": "system", "content": "Review the given code. Respond in structured JSON."},
        {"role": "user", "content": "Review this code: def add(a, b): return a + b"}
    ],
    response_format=CodeReview,
)

# 4. Access the parsed data correctly
review = response.choices[0].message.parsed

print(f"Language : {review.language}")
print(f"Summary  : {review.summary}")
print(f"Issues   : {review.issues}")
print(f"Severity : {review.severity}")

Language : Python
Summary  : Simple function to add two numbers.
Issues   : ["Function name does not follow PEP 8 conventions (should be 'add', not 'add').", 'Variable names do not indicate the purpose of a or b.', 'No error checking or handling for potential inputs.']
Severity : Low to Moderate


In [15]:
df = pd.DataFrame([review.model_dump()])
df

,language,summary,issues,severity
0,Python,Simple function to add two numbers.,[Function name does not follow PEP 8 conventio...,Low to Moderate


In [16]:
# Export to CSV for record-keeping
df.to_csv("code_review_results.csv", index=False)
print("Saved to code_review_results.csv")

Saved to code_review_results.csv


In [26]:
# import json

# Step 1: Define the tool schema
tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "Get the current weather for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and state, e.g. 'San Francisco, CA'",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                },
            },
            "required": ["location", "unit"],
        },
    }
]

In [28]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  
)

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"},
            },
            "required": ["location"],
        },
    }
}]


print(" Consulting Llama 3.2 about Lahore...")
response = client.responses.create(
    model="llama3.2",
    tools=tools,
    input="What's the weather like in Lahore?",
    tool_choice="auto",
)

for item in response.output:
    if item.type == "function_call":
        print(" TOOL TRIGGERED!")
        print(f"Function : {item.name}")
        print(f"Arguments: {item.arguments}")
    elif item.type == "message":
        # Accessing the text from the content list
        print(f"Message  : {item.content[0].text}")

 Consulting Llama 3.2 about Lahore...
 TOOL TRIGGERED!
Function : None
Arguments: {"lang":"en","api_key":"\u003cyour_api_key\u003e"}


In [29]:
# When the input doesn't need a tool, the model responds normally
response = client.responses.create(
    model="llama3.2",
    tools=tools,
    input="Hi, how are you?",
    tool_choice="auto",
)


In [30]:
print(response.output_text) 

In [31]:
response = client.responses.create(
    model="llama3.2",
    input="A farmer has 17 sheep. All but 9 run away. How many sheep does he have left?",
    reasoning={
        "effort": "high"  # Options: "low", "medium", "high"
    },
)

print(response.output_text)

I think there may be a trick question here!

The problem states that "All but 9" run away, which means the number of sheep that run away is 17 - 9 = 8.

However, if all but 9 run away, that implies that the farmer still has the original group of sheep minus 9. So, we need to subtract only 9 from the total number of sheep.

In this case, the farmer has 17 - 9 = 8 sheep left, and 9 sheep ran away.


In [32]:
response = client.responses.create(
    model="llama3.2",
    tools=[{"type": "web_search_preview"}],
    input="What are the top AI research papers this week?",
)

print(response.output_text)

In [30]:
import base64
from openai import OpenAI

# 1. Setup the client for your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# 2. Use the absolute path you provided (using 'r' for Raw String)
image_path = r"C:\Users\it king\Projects\agent_lens\notebook\134080927587012741.jpg"

try:
    # Read and encode the image
    with open(image_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")
    
    # 3. Send to Llama 3.2 (Chat Completions)
    response = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Describe this image in detail for my agent project."},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
                    },
                ],
            }
        ],
    )

    # 4. Print the result
    print("✅ Success! Model Response:")
    print("-" * 30)
    print(response.choices[0].message.content)

except FileNotFoundError:
    print(f"❌ Error: The file was not found at {image_path}. Check if the filename is 100% correct.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

✅ Success! Model Response:
------------------------------
I'm happy to help you with your agent project, but I don't see the image [img-0]. Could you please provide more context or describe the image yourself? This will allow me to assist you in describing it in detail.

If you're unable to share the actual image, I can also ask some questions to try and understand what kind of image it might be. For example:

* Is it a photo or illustration?
* What is the subject matter of the image (e.g. landscape, cityscape, object, animal, etc.)?
* Are there any specific colors or lighting effects that stand out in the image?

Please let me know and I'll do my best to help you describe the image for your agent project.


In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# 1. Load the .env file where your key is stored
load_dotenv()

# 2. Get the key from the environment variable
api_key = os.getenv("OPENAI_API_KEY")

# 3. Double check if the key actually loaded
if not api_key or api_key == "test":
    print("❌ Error: Valid API Key not found in .env file!")
else:
    client = OpenAI(api_key=api_key)
    
    # Now run your Vector Store code
    try:
        vector_store = client.vector_stores.create(name="AgentLens_Knowledge")
        print(f"✅ Success! Vector Store Created: {vector_store.id}")
    except Exception as e:
        print(f"❌ API Error: {e}")

✅ Success! Vector Store Created: vs_69d5ea795ffc81919d817af2d86e6f54


In [7]:
import os

search_path = r"C:\Users\it king"
target_filename_part = "knowledge"

print(f"Searching for PDF files containing '{target_filename_part}'...")
for root, dirs, files in os.walk(search_path):
    # Skip the .venv and .git folders to save time
    if '.venv' in root or '.git' in root:
        continue
        
    for file in files:
        if target_filename_part.lower() in file.lower() and file.endswith(".pdf"):
            full_path = os.path.join(root, file)
            print(f"🎯 FOUND IT: {full_path}")

Searching for PDF files containing 'knowledge'...


In [8]:
import base64
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. Use the absolute path we found earlier to avoid FileNotFoundError
image_path = r"C:\Users\it king\Projects\agent_lens\notebook\134080927587012741.jpg"

def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

try:
    base64_image = encode_image(image_path)

    # 2. Use client.chat.completions.create (The standard method)
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Current standard mini model with vision
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "What is in this image?"},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        },
                    },
                ],
            }
        ],
        max_tokens=300,
    )

    print(response.choices[0].message.content)

except FileNotFoundError:
    print(f"❌ Error: Could not find the image at {image_path}")
except Exception as e:
    print(f"❌ An error occurred: {e}")

❌ An error occurred: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [ ]:
import base64
import requests
import io
from PIL import Image
from openai import OpenAI

# Initialize client
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 1. Download with a User-Agent header
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(image_url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Resize to reduce RAM usage for your 8GB system
    img = Image.open(io.BytesIO(response.content))
    img = img.resize((384, 384)) 

    # Convert to Base64
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    image_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')

    # 2. Call the Ollama API
    api_response = client.chat.completions.create(
        model="qwen3.5:397b-cloud",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image."},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
            ],
        }],
    )
    print(api_response.choices[0].message.content)
else:
    print(f"Failed to download image. Status code: {response.status_code}")

This image features a close-up portrait of an orange tabby cat. Here are the specific details:

**The Cat:**
*   **Coloring:** The cat has warm, ginger-orange fur with faint darker stripes on its forehead and cheeks. There is a patch of lighter, creamy white fur on its chest and chin.
*   **Face:** It has large, amber-yellow eyes that are looking slightly off-camera to the right. Its nose is a dusty pink color, and long white whiskers fan out prominently from its muzzle.
*   **Pose:** The cat is facing forward but angled slightly to the right, with its ears perked up and attentive.

**The Background:**
*   The background is out of focus (blurred), which keeps the attention on the cat.
*   It appears to be an outdoor setting with a light gray or beige surface, likely concrete.
*   There are blurry red lines running diagonally across the top of the frame, which look like garden hoses or pipes.


In [24]:
import ollama

knowledge_base = """
SAI Company (Social Accountability International) is a global non-profit.
Our Social Media Links:
- LinkedIn: https://www.linkedin.com/company/social-accountability-international/
- Twitter/X: https://twitter.com/SAIntl
- Facebook: https://www.facebook.com/SAIntl/
- Instagram: https://www.instagram.com/sai_global/
"""

def run_local_rag():
    print("🚀 Running Local AgentLens (No API Key Required)...")
    
    try:
    
        response = ollama.chat(
            model='qwen2.5:3b',
            messages=[
                {
                    'role': 'system', 
                    'content': f'You are a RAG assistant. Use this context to answer: {knowledge_base}'
                },
                {
                    'role': 'user', 
                    'content': 'What social media links does the SAI company have?'
                }
            ]
        )
        
        print("\n--- AgentLens Search Result ---")
        print(response['message']['content'])

    except Exception as e:
        print(f"❌ local Error: {e}")

if __name__ == "__main__":
    run_local_rag()

🚀 Running Local AgentLens (No API Key Required)...

--- AgentLens Search Result ---
The social media links for SA International, which is another name for SAIC (Social Accountability International), include:

1. LinkedIn: [https://www.linkedin.com/company/social-accountability-international/](https://www.linkedin.com/company/social-accountability-international/)
2. Twitter/X: [@SAIntl](https://twitter.com/SAIntl)
3. Facebook: [https://www.facebook.com/SAIntl/](https://www.facebook.com/SAIntl/)
4. Instagram: [https://www.instagram.com/sai_global/](https://www.instagram.com/sai_global/)


In [22]:
ollama_client = OpenAI(
    api_key=os.environ.get("OLLAMA_API_KEY", "ollama"),  # Ollama ignores the key
    base_url=os.environ.get("BASE_URL", "http://localhost:11434/v1"),
)

model_name = os.environ.get("LLM_MODEL", "llama3.2")

In [23]:
import ollama
from pydantic import BaseModel
from typing import List

# 1. Define the structure (This is what was missing!)
class CodeReview(BaseModel):
    language: str
    suggestions: List[str]
    is_efficient: bool
    rating: int

model_name = "llama3.2"

try:
    print(f"🚀 {model_name} is reviewing your code...")
    
    # Using the .chat() method with 'format' for Ollama
    # Note: As of early 2026, Ollama uses the 'format' parameter for JSON schemas
    response = ollama.chat(
        model=model_name,
        messages=[
            {'role': 'system', 'content': 'Review the given code. Respond ONLY in JSON.'},
            {'role': 'user', 'content': 'Review this code: def add(a, b): return a + b'}
        ],
        format=CodeReview.model_json_schema(), # This sends the structure to the model
    )

    # 3. Parse the output
    import json
    content = json.loads(response['message']['content'])
    print("\n--- Code Review Result ---")
    print(f"Language : {content['language']}")
    print(f"Rating   : {content['rating']}/10")
    print(f"Tips     : {content['suggestions']}")

except Exception as e:
    print(f"❌ Error: {e}")

🚀 llama3.2 is reviewing your code...

--- Code Review Result ---
Language : Python
Rating   : 8/10
Tips     : []
